# Stage 1 — Preprocessing Pipeline

**Purpose**: Verify that each preprocessing step works correctly and understand the visual effect of each transformation on real MVTec images.

**What this notebook covers**:
1. Load a `good` and a `defective` image
2. Step-by-step visualization: resize → denoise → normalize
3. Augmentation: horizontal flip
4. End-to-end `preprocess()` pipeline
5. Design decisions and their rationale

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from src.preprocessing import load_image, normalize, denoise, augment, preprocess, DEFAULT_SIZE

# Paths to sample images
GOOD_PATH    = '../data/mvtec_ad/metal_nut/train/good/000.png'
DEFECT_PATH  = '../data/mvtec_ad/metal_nut/test/bent/000.png'

print('Imports OK')
print(f'Default size: {DEFAULT_SIZE}')

## 1. Load raw images

**Design decision — BGR → RGB conversion**:  
OpenCV reads images in BGR order by default. We convert to RGB immediately in `load_image()` so that all downstream code (matplotlib, torchvision) works in the same color space. Mixing BGR and RGB is a silent bug that produces wrong colors without raising any error.

In [ ]:
good_raw   = load_image(GOOD_PATH)
defect_raw = load_image(DEFECT_PATH)

print(f'Good image   — shape: {good_raw.shape}, dtype: {good_raw.dtype}, range: [{good_raw.min()}, {good_raw.max()}]')
print(f'Defect image — shape: {defect_raw.shape}, dtype: {defect_raw.dtype}, range: [{defect_raw.min()}, {defect_raw.max()}]')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(good_raw);   axes[0].set_title('Good (raw, 224x224)');    axes[0].axis('off')
axes[1].imshow(defect_raw); axes[1].set_title('Defective (raw, 224x224)'); axes[1].axis('off')
plt.suptitle('Step 1 — Load & Resize', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/results/preprocessing_step1_load.png', dpi=120)
plt.show()

## 2. Denoise

**Design decision — Gaussian blur, ksize=3**:  
Industrial camera images contain sensor noise (random pixel-level variations). Gaussian blur with a 3×3 kernel smooths this noise without blurring the texture boundaries that HOG relies on. A larger kernel (e.g., 7×7) would remove noise more aggressively but also blur defect edges, making them harder to detect.

We chose Gaussian over median because the dominant noise in MVTec images is Gaussian sensor noise, not salt-and-pepper impulse noise.

In [ ]:
good_denoised   = denoise(good_raw)
defect_denoised = denoise(defect_raw)

# Show the difference between original and denoised
diff_good   = np.abs(good_raw.astype(int)   - good_denoised.astype(int)).astype(np.uint8)
diff_defect = np.abs(defect_raw.astype(int) - defect_denoised.astype(int)).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0,0].imshow(good_raw);       axes[0,0].set_title('Good — original')
axes[0,1].imshow(good_denoised);  axes[0,1].set_title('Good — denoised (k=3)')
axes[0,2].imshow(diff_good * 10); axes[0,2].set_title('Difference ×10')
axes[1,0].imshow(defect_raw);       axes[1,0].set_title('Defective — original')
axes[1,1].imshow(defect_denoised);  axes[1,1].set_title('Defective — denoised (k=3)')
axes[1,2].imshow(diff_defect * 10); axes[1,2].set_title('Difference ×10')
for ax in axes.flat: ax.axis('off')
plt.suptitle('Step 2 — Gaussian Denoise (ksize=3)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/results/preprocessing_step2_denoise.png', dpi=120)
plt.show()

print(f'Mean absolute difference (good):    {diff_good.mean():.4f} / 255')
print(f'Mean absolute difference (defect):  {diff_defect.mean():.4f} / 255')

## 3. Normalize

**Design decision — [0, 1] range, not zero-mean**:  
We normalize to `[0.0, 1.0]` here, not to ImageNet mean/std. This keeps `preprocessing.py` reusable for the classical HOG pipeline, which does not use PyTorch. ImageNet normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`) is applied later inside the PyTorch `DataLoader` transform, only for the deep learning pipeline.

In [ ]:
good_norm   = normalize(good_denoised)
defect_norm = normalize(defect_denoised)

print(f'Good   — dtype: {good_norm.dtype}, min: {good_norm.min():.4f}, max: {good_norm.max():.4f}, mean: {good_norm.mean():.4f}')
print(f'Defect — dtype: {defect_norm.dtype}, min: {defect_norm.min():.4f}, max: {defect_norm.max():.4f}, mean: {defect_norm.mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(good_norm.ravel(),   bins=50, color='steelblue', alpha=0.7, label='good')
axes[0].set_title('Pixel distribution — Good'); axes[0].set_xlabel('Value [0,1]')
axes[1].hist(defect_norm.ravel(), bins=50, color='tomato',    alpha=0.7, label='defective')
axes[1].set_title('Pixel distribution — Defective'); axes[1].set_xlabel('Value [0,1]')
plt.suptitle('Step 3 — Normalize to [0, 1]', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/results/preprocessing_step3_normalize.png', dpi=120)
plt.show()

## 4. Augmentation

**Design decision — horizontal flip only**:  
Metal nuts are left-right symmetric, so a horizontally flipped image is a valid training sample. Vertical flip is disabled by default because nuts have a defined orientation (the thread direction). This doubles the effective training set size at zero annotation cost.

More aggressive augmentation (rotation, color jitter) is handled in the PyTorch `DataLoader` with GPU acceleration — not here.

In [ ]:
aug_good = augment(good_norm, flip_h=True, flip_v=False)

print(f'Number of augmented copies: {len(aug_good)} (original + horizontal flip)')

fig, axes = plt.subplots(1, len(aug_good), figsize=(6 * len(aug_good), 5))
labels = ['Original', 'Horizontal flip']
for i, (img, label) in enumerate(zip(aug_good, labels)):
    axes[i].imshow(img)
    axes[i].set_title(label)
    axes[i].axis('off')
plt.suptitle('Step 4 — Augmentation', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/results/preprocessing_step4_augment.png', dpi=120)
plt.show()

## 5. End-to-end pipeline

`preprocess()` chains all steps (load → denoise → normalize) in one call.  
This is the function that both the HOG+SVM and EfficientNet pipelines will call.

In [ ]:
result_good   = preprocess(GOOD_PATH)
result_defect = preprocess(DEFECT_PATH)

print('=== preprocess() output ===')
for name, img in [('Good', result_good), ('Defect', result_defect)]:
    print(f'  {name:8s} — shape: {img.shape}, dtype: {img.dtype}, '
          f'min: {img.min():.4f}, max: {img.max():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(result_good);   axes[0].set_title('Good — preprocess() output');    axes[0].axis('off')
axes[1].imshow(result_defect); axes[1].set_title('Defective — preprocess() output'); axes[1].axis('off')
plt.suptitle('End-to-end preprocess() — float32, [0,1], 224×224', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/results/preprocessing_step5_pipeline.png', dpi=120)
plt.show()

print('\nPipeline verified.')

## Summary — Design Decisions

| Step | Choice | Why |
|---|---|---|
| Color space | RGB | Matches torchvision convention; avoids silent BGR bugs |
| Resize | 224×224 | Standard ImageNet input — no architectural changes for transfer learning |
| Denoise | Gaussian, ksize=3 | Removes sensor noise without blurring defect edges |
| Normalize | [0, 1] | Shared between classical and DL pipelines; ImageNet mean/std applied later in DataLoader |
| Augmentation | Horizontal flip only | Nuts are left-right symmetric; vertical flip disabled (orientation matters) |
| Entry point | `preprocess()` | Single function for both pipelines — one place to modify if anything changes |